In [1]:
import matplotlib.pyplot as plt
import sys
sys.path.append('../')
import UTILS.utils as utils
import pandas as pd

import numpy as np
import seaborn as sns
import ast

from IPython.display import display, HTML, Image

import math
import plotly.express as px
from itertools import combinations

In [2]:
all_sentences = pd.read_csv("../../Data/TOPIC_MODELLING/ALL_SENTENCES_MULTIPLE_CATEGORIES.csv")

In [3]:
topics = utils.get_list_topics()
topic_to_id = dict()
for topic in topics:
    s = all_sentences[all_sentences["paraph_class_name"] == topic]
    topic_to_id[topic] = s["paraph_class"].sample(1).values[0]
id_to_topic = {v: k for k, v in topic_to_id.items()}
id_to_topic = dict(sorted(id_to_topic.items()))

In [5]:
df = all_sentences[all_sentences["length_cats"] > 0] 
df["categories"] = df["categories"].apply(lambda x : ast.literal_eval(x))
df["categories_probs"] = df["categories_probs"].apply(lambda x : ast.literal_eval(x))

/var/folders/yl/114pstb53sj0y8fd5bqt753r0000gn/T/ipykernel_40408/2008905728.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["categories"] = df["categories"].apply(lambda x : ast.literal_eval(x))
/var/folders/yl/114pstb53sj0y8fd5bqt753r0000gn/T/ipykernel_40408/2008905728.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["categories_probs"] = df["categories_probs"].apply(lambda x : ast.literal_eval(x))


In [8]:
df.columns

Index(['chapter', 'paragraph', 'paraph_class', 'paraph_class_name', 'text',
       'sentence_paraph_class', 'logprob', 'categories', 'categories_probs',
       'length_cats'],
      dtype='object')

In [20]:
topics

['Agriculture and Farming',
 'Architecture and Building',
 'British Colonialism',
 'Cemeteries and Burial Practices',
 'Daily Life and Occupations',
 'Education',
 'Family and Social Structures',
 'Fashion and Dress',
 'Folk Festivals and Celebrations',
 'Food and Diet',
 'Gambling and Opium',
 'Geography and Topography',
 'Harbor and Waterways',
 'Military and Naval Defenses',
 'Missionary Work',
 'Photography and Imaging',
 'Plague and Disease',
 'Prisoners and Punishment',
 'Religion (Buddhism and Ancestor Worship)',
 'River Dwellers (Tankia)',
 'Social Classes and Customs',
 'Tea Production and Trade',
 'Trade and Commerce',
 'Transportation',
 'War and Conflict']

In [21]:
new_topics = [
    "Agriculture",
    "Architecture",
    "Colonialism",
    "Cemeteries",
    "Daily Life",
    "Education",
    "Family Structures",
    "Fashion",
    "Celebrations",
    "Food",
    "Gambling",
    "Geography",
    "Waterways",
    "Military Defenses",
    "Missionary Work",
    "Photography",
    "Disease",
    "Punishment",
    "Religion",
    "River Dwellers",
    "Social Customs",
    "Tea",
    "Commerce",
    "Transportation",
    "War"
]

old_to_new_topics = dict()
for i, topic in enumerate(topics):
    old_to_new_topics[topic] = new_topics[i]

In [22]:
def get_topic_intersection(row, topics : list[int]):
    for topic in topics:
        topic_string = id_to_topic[topic]
        if topic_string not in row["categories"]:
            return False
    return True

def get_topic_intersection_summed_probs(row, topics:list[int]):
    summed = 0
    for topic in topics:
        topic_string = id_to_topic[topic]
        topic_index = row["categories"].index(topic_string)
        summed += row["categories_probs"][topic_index]
    return summed / len(topics)

def get_sentences_intersection(all_sentences : pd.DataFrame, topics : list[int]):
    mask = all_sentences.apply(lambda x : get_topic_intersection(x, topics), axis=1)
    selected_sentences = all_sentences[mask].copy()  # safer to copy!
    if(len(selected_sentences) == 0):
        return ([], [])
    summed_prob = selected_sentences.apply(
    lambda x: float(get_topic_intersection_summed_probs(x, topics)),
    axis=1
    )
    selected_sentences["summed_prob"] = summed_prob.values
    selected_sentences = selected_sentences.sort_values("summed_prob", ascending=False)
    sentences = list(selected_sentences["text"].values)

    images = list(set(selected_sentences["chapter"].to_list())) 
    return (sentences, images)


In [34]:
from tqdm import tqdm

all_combinations = [list(combinations(id_to_topic.keys(), r)) for r in range(1, 5)]
all_combinations = [i for s in all_combinations for i in s]

length_counts_images = dict()
length_counts_sentences = dict()
combis = dict()

for combi in tqdm(all_combinations):
    (sentences, images) = get_sentences_intersection(df, list(combi))
    combis[str(combi)] = len(sentences)
    if len(sentences) not in length_counts_sentences.keys():
        length_counts_sentences[len(sentences)] = 1
    else:
        length_counts_sentences[len(sentences)] += 1
    if len(images) not in length_counts_images.keys():
        length_counts_images[len(images)] = 1
    else:
        length_counts_images[len(images)] += 1

100%|██████████| 15275/15275 [01:19<00:00, 192.61it/s]


In [33]:
import json
file_path = 'combis.json'
with open(file_path, 'w') as file:
    json.dump(combis, file, indent=4) # indent=4 makes it pretty-print


In [9]:
import plotly.express as px
fig = px.bar(x = length_counts_images.keys(), y = length_counts_images.values())
fig.show()

In [10]:
import tkinter as tk
from functools import partial
from tkinter import DISABLED, NORMAL
from PIL import ImageTk, Image
from random import sample


class Mockup():
    
    def __init__(self):
        self.root = tk.Tk()
        self.root.geometry('1800x1000')

        self.images= []
        self.image_panels = []

        self.imageFrame = tk.Frame(self.root)
        for i in range(5):
            self.imageFrame.columnconfigure(i, weight=1, pad=10)
            self.images.append(self.open_image(i+1))

        for i in range(5):
            panel = tk.Label(self.imageFrame, image = self.images[0], padx=20)
            panel.grid(row=0, column=i)
            self.image_panels.append(panel)

        self.imageFrame.pack(padx=10, pady=10)


        self.text = []
        for i in range(8):
            label = tk.Label(self.root, text=df.sample(1)["text"].values[0], font=("Arial", 12), wraplength=1300)
            label.pack(padx=5, pady=5)
            self.text.append(label)

        self.selectedTopics = []


        button = tk.Button(self.root, text = "Show text & images", command=self.button_clicked)
        button.pack(padx=10, pady=10)


        self.checkboxframe = tk.Frame(self.root)
        for i in range(3):
            self.checkboxframe.columnconfigure(i, weight=1)

        self.checkboxes = []
        self.check_states = []
        for i, topic in enumerate(topics):
            check_state = tk.IntVar()
            checkbox = tk.Checkbutton(self.checkboxframe, text = old_to_new_topics[topic], font=("Arial", 16), variable=check_state, anchor="w", command=partial(self.disableTopicsWithNoIntersection, topic, i))

            row = int(i / 3)
            column = i % 3

            checkbox.grid(row=row, column=column, sticky=tk.W+tk.E)
            self.checkboxes.append(checkbox)
            self.check_states.append(check_state)
        
        self.checkboxframe.pack(padx=10, pady=10, fill="x", side="bottom")
        #self.root.attributes("-fullscreen", True)

        self.root.mainloop()

    def disableTopicsWithNoIntersection(self, topic_name, state_index):
        if(self.check_states[state_index].get() == 0):
            self.selectedTopics.remove(topic_to_id[topic_name])
            for i, topic in enumerate(topics):
                new_list = self.selectedTopics + [topic_to_id[topic]]
                if len(get_sentences_intersection(df, new_list)[0]) > 0:
                    self.checkboxes[i].config(state=NORMAL)
        else:
            self.selectedTopics.append(topic_to_id[topic_name])
            for i, topic in enumerate(topics):
                new_list = self.selectedTopics + [topic_to_id[topic]]
                if len(get_sentences_intersection(df, new_list)[0]) == 0:
                    self.checkboxes[i].config(state=DISABLED)

    def open_image(self, image_num):
        img = Image.open("../../Data/STEREO_VIEWS/LEFT_IMAGE/image_" + str(image_num+1) + ".png")
        img = img.resize((250, 250))
        img = ImageTk.PhotoImage(img)
        return img

    def button_clicked(self):
        (sentences, images) = get_sentences_intersection(df, self.selectedTopics)
        print("Query has " + str(len(sentences)) + " sentences and " + str(len(images)) + " images.")
        for sentence in sentences :
            print(sentence)
        for image in images:
            print(image)
        if len(sentences) > 8:
            sentences = list(sentences[:8])
        else:
            sentences = list(sentences) + [""] * (8 - len(sentences))
        if len(images) > 5:
            images = sample(images, 5)
        for i,text in enumerate(self.text):
            text.config(text=sentences[i])
        for i, image in enumerate(self.image_panels):
            if(i >= len(images)):
                image.config(image="")
                image.photo = ""
            else:
                new_img = self.open_image(list(images)[i])
                image.config(image=new_img)
                image.photo = new_img

Mockup()

: 